# Módulo 41: Importación y consultas SQL — Solución

Author: Juliho David Castillo Colmenares

Enunciado para estudiantes: [m41-importacion-y-consultas-sql-guia.ipynb](m41-importacion-y-consultas-sql-guia.ipynb) · Lección: [m41-importacion-y-consultas-sql-leccion.ipynb](m41-importacion-y-consultas-sql-leccion.ipynb)

## Contexto

El área de Recursos Humanos de una empresa quiere entender la rotación de personal antes de diseñar un plan de retención. Se importan los datos de empleados desde un CSV a una base de datos SQLite creada con Python y se responden varias preguntas con consultas SQL que combinan filtros y funciones de agregación.

## Datos

`recursos-humanos.csv` (en esta carpeta; fuente: [JoseRaulCastro/EBAC](https://github.com/JoseRaulCastro/EBAC/blob/main/recursos_humanos.csv)): 14 999 empleados con `satisfaction_level`, `last_evaluation`, `number_project`, `average_montly_hours` (con errata), `time_spend_company`, `Work_accident`, `left`, `promotion_last_5years`, `sales` (departamento) y `salary` (`low`, `medium`, `high`).

## Instrucciones

1. **Crear la base `RH.db` con la tabla `Detalle`** que importe el CSV (con la columna de horas renombrada a `average_monthly_hours`).
2. **Satisfacción promedio** de todos los empleados.
3. **¿Quiénes están más satisfechos?** Satisfacción promedio de quienes abandonan la empresa (`left = 1`) frente a quienes se quedan (`left = 0`).
4. **Horas mensuales con salario bajo o medio:** promedio de `average_monthly_hours` con `salary` `low` o `medium`.
5. **Promovidos que se fueron:** registros con `promotion_last_5years = 1` que abandonaron la empresa.
6. **Evaluaciones altas:** registros con `last_evaluation` de 0.9 o más.

Entrega: el notebook y un PDF con capturas del código y los resultados.

### Carga de datos

In [1]:
import sqlite3
import pandas

In [2]:
# Create a datafile for the database
data = pandas.read_csv("recursos-humanos.csv")
data.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,sales,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


### Tarea 1. Crear la base `RH.db` con la tabla `Detalle`

In [3]:
# Create a new SQLite database and establish a connection
conn = sqlite3.connect("RH.db")
cursor = conn.cursor()

In [4]:
# Insert the CSV data into the 'Detalle' table
data.columns = [
    col.replace("average_montly_hours", "average_monthly_hours").lower()
    for col in data.columns
]
data.to_sql("Detalle", conn, if_exists="replace", index=False)

14999

### Tarea 2. Satisfacción promedio

In [5]:
# 1. Calculate the average satisfaction level of all employees
cursor.execute("SELECT AVG(satisfaction_level) FROM Detalle")
avg_satisfaction = cursor.fetchone()[0]
avg_satisfaction

0.6128335222348156

### Tarea 3. ¿Quiénes están más satisfechos?

In [6]:
# 2. Compare satisfaction levels between employees who stayed and those who left
cursor.execute("SELECT AVG(satisfaction_level) FROM Detalle WHERE left = 1")
avg_satisfaction_left = cursor.fetchone()[0]
avg_satisfaction_left

0.44009801176141133

In [7]:
cursor.execute("SELECT AVG(satisfaction_level) FROM Detalle WHERE left = 0")
avg_satisfaction_stayed = cursor.fetchone()[0]
avg_satisfaction_stayed

0.666809590479524

### Tarea 4. Horas mensuales con salario bajo o medio

In [8]:
# 3. Calculate the average monthly hours for employees with low or medium salary
cursor.execute(
    "SELECT AVG(average_monthly_hours) FROM Detalle WHERE salary IN ('low', 'medium')"
)
avg_hours_low_medium_salary = cursor.fetchone()[0]
avg_hours_low_medium_salary

201.15666327568667

### Tarea 5. Promovidos que se fueron

In [9]:
# 4. Extract records of employees promoted in the last 5 years who left the company
cursor.execute("SELECT * FROM Detalle WHERE promotion_last_5years = 1 AND left = 1")
promoted_left = cursor.fetchall()
promoted_left[:5]

[(0.45, 0.51, 2, 160, 3, 1, 1, 1, 'sales', 'low'),
 (0.79, 0.59, 4, 139, 3, 0, 1, 1, 'management', 'low'),
 (0.41, 0.46, 2, 160, 3, 0, 1, 1, 'sales', 'low'),
 (0.11, 0.79, 6, 292, 4, 0, 1, 1, 'technical', 'low'),
 (0.41, 0.56, 2, 154, 3, 0, 1, 1, 'support', 'medium')]

### Tarea 6. Evaluaciones altas

In [10]:
# 5. Extract records of employees with a last evaluation of 0.9 or higher
cursor.execute("SELECT * FROM Detalle WHERE last_evaluation >= 0.9")
high_evaluation = cursor.fetchall()
high_evaluation[:5]

[(0.89, 1.0, 5, 224, 5, 0, 1, 0, 'sales', 'low'),
 (0.84, 0.92, 4, 234, 5, 0, 1, 0, 'sales', 'low'),
 (0.78, 0.99, 4, 255, 6, 0, 1, 0, 'sales', 'low'),
 (0.09, 0.95, 6, 304, 4, 0, 1, 0, 'sales', 'low'),
 (0.89, 0.92, 5, 242, 5, 0, 1, 0, 'sales', 'low')]

### Cierre y resumen de resultados

In [11]:
# Close the connection
conn.close()

In [12]:
# Display the results to the user
print(f"Average Satisfaction Level: {avg_satisfaction:.2f}")
print(f"Average Satisfaction Level (Left): {avg_satisfaction_left:.2f}")
print(f"Average Satisfaction Level (Stayed): {avg_satisfaction_stayed:.2f}")
print(f"Average Monthly Hours (Low/Medium Salary): {avg_hours_low_medium_salary:.2f}")
print(f"Number of Promoted Employees Who Left: {len(promoted_left)}")
print(f"Number of Employees with High Evaluation: {len(high_evaluation)}")

Average Satisfaction Level: 0.61
Average Satisfaction Level (Left): 0.44
Average Satisfaction Level (Stayed): 0.67
Average Monthly Hours (Low/Medium Salary): 201.16
Number of Promoted Employees Who Left: 19
Number of Employees with High Evaluation: 2988


**Revisión posterior (2026-10):** la base de datos se llama ahora `RH.db` y la tabla `Detalle`, como pide el enunciado (antes `data.db` y `Details`), y se quitó el `CREATE TABLE` manual, que `to_sql(if_exists="replace")` sobrescribía. El PDF conserva la versión entregada.